# Configuration & Setup

In [1]:
"""
================================================================================
PROJECT: ENTERPRISE SECURITY OPERATIONS CENTER (SOC) SIMULATOR
================================================================================

AUTHOR: Prashant Ranjan
COURSE: Google Cybersecurity Certificate
INTERNSHIP: Ethical Hacking (8 Weeks)
DATE: August 2026

================================================================================
PROJECT OBJECTIVES
================================================================================

1. Design a simulated Security Operations Center environment
2. Implement SIEM-like log collection and analysis
3. Apply threat detection rules using SQL and Python
4. Demonstrate Incident Response procedures
5. Generate security analytics and visualizations
6. Produce executive-level security reports

================================================================================
SCOPE AND LIMITATIONS
================================================================================

This project is an educational simulation. All data is synthetically generated
and does not represent real organizational information.

Key limitations:
- No live network packet capture
- No connection to external security tools
- No real-time data ingestion
- No integration with Active Directory or IAM systems
- All alerts are rule-based and simulated
- Not intended for production deployment

================================================================================
ASSUMPTIONS
================================================================================

1. All data is synthetically generated for demonstration purposes
2. IP addresses follow RFC 5737 documentation ranges
3. Password hashes are generated for demonstration only
4. No real organizational data is used
5. All attack patterns are simulated
6. Detection rules are simplified for educational clarity
7. Heuristic scores are calculated for demonstration purposes

================================================================================
VERSION HISTORY
================================================================================

v1.0 - Database schema design
v1.5 - Data generation and population
v2.0 - Detection rule implementation
v2.5 - Incident response logic
v3.0 - Dashboard and visualization
v3.5 - Executive reporting
v4.0 - Educational simulator with logging, testing, error handling

================================================================================
ARCHITECTURE OVERVIEW
================================================================================

Data Sources (Synthetic)
    |
    v
SQLite Database (10 Tables)
    |
    v
Threat Detection Engine (5 Rules)
    |
    v
Alert Correlation
    |
    v
Incident Response (4 Playbooks)
    |
    v
Visualization Dashboard (9 Charts)
    |
    v
Executive Report
================================================================================
"""

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import hashlib
import re
import json
import random
import logging
import time
import unittest
import sys
from typing import Dict, List, Tuple, Optional, Any
import warnings
warnings.filterwarnings('ignore')

# Attempt to import bcrypt with fallback
try:
    import bcrypt
    BCRYPT_AVAILABLE = True
except ImportError:
    BCRYPT_AVAILABLE = False
    print("Note: bcrypt not available - using simple hash for demonstration")

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    "ASSETS": 50,
    "USERS": 50,
    "AUTH_LOGS": 10000,
    "NETWORK_EVENTS": 5000,
    "MALWARE_EVENTS": 250,
    "VULNERABILITIES": 100,
    "FIREWALL_LOGS": 1000,
    "BRUTE_FORCE_THRESHOLD": 5,
    "PORT_SCAN_THRESHOLD": 20,
    "RANDOM_SEED": 42,
    "LOG_LEVEL": "INFO"
}

# ============================================================================
# LOGGING SETUP
# ============================================================================
logging.basicConfig(
    level=getattr(logging, CONFIG["LOG_LEVEL"]),
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# ============================================================================
# PROJECT INITIALIZATION
# ============================================================================
logger.info("=" * 80)
logger.info("ENTERPRISE SECURITY OPERATIONS CENTER (SOC) SIMULATOR")
logger.info("=" * 80)
logger.info(f"Session initialized: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info(f"Analyst: Prashant Ranjan")
logger.info(f"Configuration: {json.dumps(CONFIG, indent=2)}")
logger.info("=" * 80)

# Set random seed for reproducibility
np.random.seed(CONFIG["RANDOM_SEED"])
random.seed(CONFIG["RANDOM_SEED"])

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

2026-08-01 12:53:56 - INFO - ================================================================================
2026-08-01 12:53:56 - INFO - ENTERPRISE SECURITY OPERATIONS CENTER (SOC) SIMULATOR
2026-08-01 12:53:56 - INFO - ================================================================================
2026-08-01 12:53:56 - INFO - Session initialized: 2026-08-01 12:53:56
2026-08-01 12:53:56 - INFO - Analyst: Prashant Ranjan
2026-08-01 12:53:56 - INFO - Configuration: {
  "ASSETS": 50,
  "USERS": 50,
  "AUTH_LOGS": 10000,
  "NETWORK_EVENTS": 5000,
  "MALWARE_EVENTS": 250,
  "VULNERABILITIES": 100,
  "FIREWALL_LOGS": 1000,
  "BRUTE_FORCE_THRESHOLD": 5,
  "PORT_SCAN_THRESHOLD": 20,
  "RANDOM_SEED": 42,
  "LOG_LEVEL": "INFO"
}
2026-08-01 12:53:56 - INFO - ================================================================================


Note: bcrypt not available - using simple hash for demonstration


# Database Architecture

In [2]:
"""
================================================================================
MODULE 1: DATABASE ARCHITECTURE
================================================================================

OBJECTIVE:
Design a relational database schema to store security event data.

METHODOLOGY:
The schema follows normalization principles and includes tables for assets,
users, authentication logs, network traffic, firewall logs, malware events,
vulnerabilities, alerts, incidents, and playbooks.

FRAMEWORK MAPPING:
- NIST CSF: Identify (Asset Management)
- CISSP Domain: Asset Security, IAM, Security Operations

EXPECTED OUTPUT:
A SQLite database with 10 interconnected tables.
================================================================================
"""

logger.info("\n" + "=" * 80)
logger.info("MODULE 1: DATABASE ARCHITECTURE")
logger.info("=" * 80)

# Initialize database connection with error handling
try:
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    logger.info("Database connection established successfully")
except sqlite3.Error as e:
    logger.error(f"Database connection failed: {e}")
    raise

# Create database schema with error handling
try:
    cursor.executescript('''
    -- Table: assets (CISSP Domain 2: Asset Security)
    CREATE TABLE assets (
        asset_id INTEGER PRIMARY KEY,
        asset_name TEXT NOT NULL,
        asset_type TEXT CHECK(asset_type IN ('Server', 'Workstation', 'Network Device',
                                             'Database', 'Application', 'Cloud')),
        ip_address TEXT,
        operating_system TEXT,
        department TEXT,
        criticality TEXT CHECK(criticality IN ('Critical', 'High', 'Medium', 'Low')),
        owner TEXT,
        patch_status TEXT CHECK(patch_status IN ('Up-to-date', 'Needs Update',
                                                'Critical Patch Required')),
        last_scan DATE,
        notes TEXT
    );

    -- Table: users (CISSP Domain 5: IAM)
    CREATE TABLE users (
        user_id INTEGER PRIMARY KEY,
        username TEXT UNIQUE NOT NULL,
        email TEXT NOT NULL,
        full_name TEXT,
        department TEXT,
        role TEXT CHECK(role IN ('Administrator', 'Security Analyst', 'Developer',
                                'Manager', 'Employee', 'Intern')),
        access_level TEXT CHECK(access_level IN ('Admin', 'Write', 'Read', 'Read-Only')),
        mfa_enabled BOOLEAN DEFAULT 0,
        created_date DATE,
        last_login DATETIME,
        password_hash TEXT,
        account_locked BOOLEAN DEFAULT 0,
        failed_attempts INTEGER DEFAULT 0
    );

    -- Table: auth_logs (NIST CSF: Detect)
    CREATE TABLE auth_logs (
        log_id INTEGER PRIMARY KEY,
        user_id INTEGER,
        username TEXT,
        source_ip TEXT,
        country TEXT,
        browser TEXT,
        operating_system TEXT,
        auth_type TEXT CHECK(auth_type IN ('Password', 'MFA', 'SSO', 'Certificate')),
        status TEXT CHECK(status IN ('SUCCESS', 'FAILED', 'LOCKED')),
        timestamp DATETIME,
        response_time_ms INTEGER,
        FOREIGN KEY (user_id) REFERENCES users(user_id)
    );

    -- Table: network_traffic (CISSP Domain 4: Network Security)
    CREATE TABLE network_traffic (
        traffic_id INTEGER PRIMARY KEY,
        src_ip TEXT,
        dst_ip TEXT,
        src_port INTEGER,
        dst_port INTEGER,
        protocol TEXT CHECK(protocol IN ('TCP', 'UDP', 'ICMP', 'HTTP', 'HTTPS',
                                         'DNS', 'SSH', 'FTP')),
        bytes_transferred INTEGER,
        direction TEXT CHECK(direction IN ('Inbound', 'Outbound', 'Internal')),
        timestamp DATETIME,
        is_malicious BOOLEAN DEFAULT 0,
        threat_type TEXT
    );

    -- Table: firewall_logs (NIST CSF: Protect)
    CREATE TABLE firewall_logs (
        log_id INTEGER PRIMARY KEY,
        rule_name TEXT,
        src_ip TEXT,
        dst_ip TEXT,
        dst_port INTEGER,
        protocol TEXT,
        action TEXT CHECK(action IN ('Allow', 'Block', 'Drop', 'Log')),
        reason TEXT,
        timestamp DATETIME
    );

    -- Table: malware_events (CISSP Domain 7: Security Operations)
    CREATE TABLE malware_events (
        event_id INTEGER PRIMARY KEY,
        asset_id INTEGER,
        malware_name TEXT,
        malware_type TEXT CHECK(malware_type IN ('Ransomware', 'Trojan', 'Worm',
                                                'Virus', 'Spyware', 'Adware', 'Rootkit')),
        severity TEXT CHECK(severity IN ('Critical', 'High', 'Medium', 'Low')),
        detection_method TEXT CHECK(detection_method IN ('Signature', 'Heuristic',
                                                        'Behavioral', 'Sandbox')),
        file_path TEXT,
        timestamp DATETIME,
        status TEXT CHECK(status IN ('Quarantined', 'Cleaned', 'Ignored', 'Failed')),
        FOREIGN KEY (asset_id) REFERENCES assets(asset_id)
    );

    -- Table: vulnerabilities (OWASP, CISSP Domain 8)
    CREATE TABLE vulnerabilities (
        vuln_id INTEGER PRIMARY KEY,
        asset_id INTEGER,
        cve_id TEXT,
        title TEXT,
        description TEXT,
        cvss_score REAL,
        severity TEXT CHECK(severity IN ('Critical', 'High', 'Medium', 'Low')),
        owasp_category TEXT,
        exploit_available BOOLEAN DEFAULT 0,
        discovered_date DATE,
        patched_date DATE,
        status TEXT CHECK(status IN ('Open', 'In Progress', 'Patched', 'Accepted Risk')),
        FOREIGN KEY (asset_id) REFERENCES assets(asset_id)
    );

    -- Table: alerts (NIST CSF: Detect, MITRE ATT&CK)
    CREATE TABLE alerts (
        alert_id INTEGER PRIMARY KEY,
        title TEXT,
        description TEXT,
        severity TEXT CHECK(severity IN ('Critical', 'High', 'Medium', 'Low', 'Informational')),
        mitre_tactic TEXT,
        mitre_technique TEXT,
        source_type TEXT CHECK(source_type IN ('SIEM', 'IDS', 'Firewall',
                                              'Malware Scanner', 'Vulnerability Scanner')),
        timestamp DATETIME,
        status TEXT CHECK(status IN ('Open', 'Investigating', 'Contained',
                                    'Resolved', 'False Positive')),
        assigned_to TEXT,
        playbook_applied TEXT
    );

    -- Table: incidents (NIST CSF: Respond)
    CREATE TABLE incidents (
        incident_id INTEGER PRIMARY KEY,
        title TEXT,
        description TEXT,
        detection_time DATETIME,
        severity TEXT CHECK(severity IN ('Critical', 'High', 'Medium', 'Low')),
        status TEXT CHECK(status IN ('Open', 'Investigating', 'Contained',
                                    'Eradicated', 'Recovered', 'Closed')),
        lead_analyst TEXT,
        affected_assets TEXT,
        root_cause TEXT,
        containment_actions TEXT,
        recovery_actions TEXT,
        lessons_learned TEXT,
        closed_time DATETIME
    );

    -- Table: playbooks (NIST CSF: Respond)
    CREATE TABLE playbooks (
        playbook_id INTEGER PRIMARY KEY,
        name TEXT,
        description TEXT,
        alert_type TEXT,
        steps TEXT,
        roles_involved TEXT,
        tools_used TEXT,
        sla_hours INTEGER
    );
    ''')
    logger.info("Database schema created successfully with 10 tables")
except sqlite3.Error as e:
    logger.error(f"Schema creation failed: {e}")
    raise

2026-08-01 12:53:56 - INFO - 
2026-08-01 12:53:56 - INFO - MODULE 1: DATABASE ARCHITECTURE
2026-08-01 12:53:56 - INFO - ================================================================================
2026-08-01 12:53:56 - INFO - Database connection established successfully
2026-08-01 12:53:56 - INFO - Database schema created successfully with 10 tables


# Data Generation

In [3]:
"""
================================================================================
MODULE 2: DATA GENERATION
================================================================================

OBJECTIVE:
Generate realistic synthetic security data with error handling and validation.

METHODOLOGY:
Statistical distributions generate realistic patterns including:
- Normal user behavior (85% of events)
- Failed authentication attempts (12% of events)
- Malicious network traffic (5% of events)
- Vulnerability findings (random distribution)

FRAMEWORK MAPPING:
- NIST CSF: Identify (Asset Inventory), Detect (Event Generation)

EXPECTED OUTPUT:
A populated database with 16,500+ security events.
================================================================================
"""

logger.info("\n" + "=" * 80)
logger.info("MODULE 2: DATA GENERATION")
logger.info("=" * 80)

def generate_assets(count: int) -> List[Tuple]:
    """Generate asset records."""
    asset_types = ['Server', 'Workstation', 'Network Device', 'Database', 'Application', 'Cloud']
    oss = ['Windows Server 2022', 'Windows 10', 'Ubuntu 22.04', 'Red Hat 9', 'macOS 13', 'CentOS 8']
    departments = ['IT', 'Security', 'Finance', 'HR', 'Marketing', 'Sales', 'Engineering', 'Executive']
    criticalities = ['Critical', 'High', 'Medium', 'Low']

    assets = []
    for i in range(1, count + 1):
        asset = (
            i,
            f"{random.choice(['WEB','DB','APP','FW','DNS','MAIL','PROXY'])}-{i:03d}",
            random.choice(asset_types),
            f"192.168.{random.randint(1,255)}.{random.randint(1,254)}",
            random.choice(oss),
            random.choice(departments),
            random.choices(criticalities, weights=[0.2, 0.3, 0.3, 0.2])[0],
            f"user{random.randint(1,count)}@corp.com",
            random.choices(['Up-to-date', 'Needs Update', 'Critical Patch Required'],
                          weights=[0.5, 0.3, 0.2])[0],
            (datetime.now() - timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d'),
            f"Asset for {random.choice(['Production', 'Development', 'Testing'])}"
        )
        assets.append(asset)
    return assets

def generate_users(count: int) -> List[Tuple]:
    """Generate user records."""
    username_list = ['admin', 'jdoe', 'asmith', 'bjones', 'cwhite', 'dblack']
    full_name_list = ['John Doe', 'Alice Smith', 'Bob Jones', 'Carol White', 'David Black']
    departments = ['IT', 'Security', 'Finance', 'HR', 'Marketing', 'Sales', 'Engineering', 'Executive']
    roles = ['Administrator', 'Security Analyst', 'Developer', 'Manager', 'Employee', 'Intern']
    access_levels = ['Admin', 'Write', 'Read', 'Read-Only']

    users = []
    for i in range(1, count + 1):
        username = f"user{i}" if i > 6 else username_list[i-1]
        name = full_name_list[i-1] if i <= 5 else f"User {i}"
        role = random.choices(roles, weights=[0.05, 0.15, 0.15, 0.15, 0.35, 0.15])[0]
        access = random.choices(access_levels, weights=[0.05, 0.15, 0.40, 0.40])[0]

        # Generate password hash
        if BCRYPT_AVAILABLE:
            pwd_hash = bcrypt.hashpw(f"Password{random.randint(1,999)}".encode('utf-8'),
                                    bcrypt.gensalt()).decode('utf-8')
        else:
            pwd_hash = hashlib.sha256(f"Password{random.randint(1,999)}".encode()).hexdigest()

        users.append((
            i, username, f"{username}@corp.com", name,
            random.choice(departments), role, access,
            random.choice([0, 1]) if i < 30 else 1,
            (datetime.now() - timedelta(days=random.randint(30,730))).strftime('%Y-%m-%d'),
            datetime.now() - timedelta(minutes=random.randint(0, 1440)),
            pwd_hash,
            random.choice([0, 1]) if i < 45 else 0,
            random.randint(0, 10) if i < 45 else 0
        ))
    return users

# Execute data generation with timing
start_time = time.perf_counter()

try:
    # Generate and insert assets
    assets = generate_assets(CONFIG["ASSETS"])
    cursor.executemany('''
    INSERT INTO assets (asset_id, asset_name, asset_type, ip_address, operating_system,
                        department, criticality, owner, patch_status, last_scan, notes)
    VALUES (?,?,?,?,?,?,?,?,?,?,?)
    ''', assets)
    logger.info(f"Generated: {len(assets)} assets")

    # Generate and insert users
    users = generate_users(CONFIG["USERS"])
    cursor.executemany('''
    INSERT INTO users (user_id, username, email, full_name, department, role,
                       access_level, mfa_enabled, created_date, last_login,
                       password_hash, account_locked, failed_attempts)
    VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
    ''', users)
    logger.info(f"Generated: {len(users)} users")

    conn.commit()
    logger.info("Data generation completed successfully")

except sqlite3.Error as e:
    logger.error(f"Data generation failed: {e}")
    conn.rollback()
    raise

end_time = time.perf_counter()
logger.info(f"Data generation execution time: {end_time - start_time:.2f} seconds")

2026-08-01 12:53:56 - INFO - 
2026-08-01 12:53:56 - INFO - MODULE 2: DATA GENERATION
2026-08-01 12:53:56 - INFO - ================================================================================
2026-08-01 12:53:56 - INFO - Generated: 50 assets
2026-08-01 12:53:56 - INFO - Generated: 50 users
2026-08-01 12:53:56 - INFO - Data generation completed successfully
2026-08-01 12:53:56 - INFO - Data generation execution time: 0.01 seconds


# Log Generation

In [4]:
"""
================================================================================
MODULE 2B: LOG GENERATION
================================================================================

OBJECTIVE:
Generate authentication, network, malware, and firewall logs.

METHODOLOGY:
- Authentication logs: 85% success, 12% failure, 3% locked
- Network traffic: 5% malicious with threat classification
- Malware events: Distributed across types with severity weighting
- Firewall logs: Rule-based with 4 rule types

EXPECTED OUTPUT:
16,250+ log entries with realistic patterns.
================================================================================
"""

logger.info("\n" + "=" * 80)
logger.info("MODULE 2B: LOG GENERATION")
logger.info("=" * 80)

start_time = time.perf_counter()

try:
    # ============================================================================
    # AUTHENTICATION LOGS
    # ============================================================================
    countries = ['India', 'USA', 'UK', 'Canada', 'Australia', 'Germany',
                 'France', 'Japan', 'Singapore', 'Brazil']
    browsers = ['Chrome', 'Firefox', 'Safari', 'Edge', 'Opera']
    os_list = ['Windows 10', 'Windows 11', 'macOS', 'Linux', 'iOS', 'Android']
    auth_types = ['Password', 'MFA', 'SSO', 'Certificate']

    auth_logs = []
    for i in range(CONFIG["AUTH_LOGS"]):
        user_id = random.randint(1, CONFIG["USERS"])
        username = f"user{user_id}" if user_id > 6 else ['admin', 'jdoe', 'asmith', 'bjones', 'cwhite', 'dblack'][user_id-1]

        if random.random() < 0.02:  # Brute force simulation
            ip = f"{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}"
            for _ in range(random.randint(5, 20)):
                auth_logs.append((
                    len(auth_logs) + 1, user_id, username, ip, random.choice(countries),
                    random.choice(browsers), random.choice(os_list),
                    random.choice(auth_types), 'FAILED',
                    datetime.now() - timedelta(minutes=random.randint(0, 60)),
                    random.randint(10, 500)
                ))
        else:
            auth_logs.append((
                len(auth_logs) + 1, user_id, username,
                f"192.168.{random.randint(1,255)}.{random.randint(1,254)}",
                random.choice(countries), random.choice(browsers), random.choice(os_list),
                random.choice(auth_types),
                random.choices(['SUCCESS', 'FAILED', 'LOCKED'], weights=[0.85, 0.12, 0.03])[0],
                datetime.now() - timedelta(minutes=random.randint(0, 720)),
                random.randint(10, 1000)
            ))

    cursor.executemany('''
    INSERT INTO auth_logs (log_id, user_id, username, source_ip, country, browser,
                           operating_system, auth_type, status, timestamp, response_time_ms)
    VALUES (?,?,?,?,?,?,?,?,?,?,?)
    ''', auth_logs)
    logger.info(f"Generated: {len(auth_logs):,} authentication logs")

    # ============================================================================
    # NETWORK TRAFFIC
    # ============================================================================
    protocols = ['TCP', 'UDP', 'HTTP', 'HTTPS', 'DNS', 'SSH', 'FTP', 'ICMP']
    known_malicious_ips = ['203.0.113.5', '198.51.100.23', '185.130.5.253', '94.102.61.78']
    threat_types = ['Port Scan', 'Malware C2', 'Data Exfiltration', 'DDoS', 'Brute Force']

    network_traffic = []
    for i in range(CONFIG["NETWORK_EVENTS"]):
        is_malicious = random.random() < 0.05
        src_ip = random.choice(known_malicious_ips) if is_malicious else f"192.168.{random.randint(1,255)}.{random.randint(1,254)}"
        network_traffic.append((
            i+1, src_ip,
            f"10.0.{random.randint(1,255)}.{random.randint(1,254)}",
            random.randint(1024, 65535),
            random.choice([80, 443, 22, 53, 21, 3389, 25, 8080, 3306, 1433]),
            random.choice(protocols),
            random.randint(64, 65536),
            'Inbound' if '192.168' not in src_ip else 'Internal',
            datetime.now() - timedelta(minutes=random.randint(0, 1440)),
            is_malicious,
            random.choice(threat_types) if is_malicious else None
        ))

    cursor.executemany('''
    INSERT INTO network_traffic (traffic_id, src_ip, dst_ip, src_port, dst_port,
                                 protocol, bytes_transferred, direction, timestamp,
                                 is_malicious, threat_type)
    VALUES (?,?,?,?,?,?,?,?,?,?,?)
    ''', network_traffic)
    logger.info(f"Generated: {len(network_traffic):,} network traffic events")

    # ============================================================================
    # MALWARE EVENTS
    # ============================================================================
    malware_names = ['WannaCry', 'Conficker', 'Zeus', 'Emotet', 'LockBit', 'Ryuk', 'Mirai', 'Sodinokibi']
    malware_types = ['Ransomware', 'Trojan', 'Worm', 'Virus', 'Spyware', 'Adware', 'Rootkit']
    detection_methods = ['Signature', 'Heuristic', 'Behavioral', 'Sandbox']

    malware_events = []
    for i in range(CONFIG["MALWARE_EVENTS"]):
        malware_events.append((
            i+1, random.randint(1, CONFIG["ASSETS"]),
            random.choice(malware_names), random.choice(malware_types),
            random.choices(['Critical', 'High', 'Medium', 'Low'], weights=[0.2, 0.3, 0.3, 0.2])[0],
            random.choice(detection_methods),
            f"C:\\Windows\\System32\\malware_{i}.exe",
            datetime.now() - timedelta(hours=random.randint(1, 168)),
            random.choices(['Quarantined', 'Cleaned', 'Ignored', 'Failed'],
                          weights=[0.5, 0.3, 0.1, 0.1])[0]
        ))

    cursor.executemany('''
    INSERT INTO malware_events (event_id, asset_id, malware_name, malware_type,
                                severity, detection_method, file_path, timestamp, status)
    VALUES (?,?,?,?,?,?,?,?,?)
    ''', malware_events)
    logger.info(f"Generated: {len(malware_events)} malware events")

    # ============================================================================
    # VULNERABILITIES
    # ============================================================================
    cve_data = [
        ('CVE-2023-1234', 'SQL Injection Vulnerability', 9.8, 'Critical', 'A03:2021-Injection'),
        ('CVE-2023-5678', 'Cross-Site Scripting (XSS)', 7.2, 'High', 'A07:2021-Cross-Site Scripting'),
        ('CVE-2023-9012', 'Weak Authentication Mechanism', 6.5, 'Medium', 'A07:2021-Identification'),
        ('CVE-2023-3456', 'Insecure Deserialization', 8.1, 'High', 'A08:2021-Integrity Failures'),
        ('CVE-2023-7890', 'Security Misconfiguration', 5.3, 'Medium', 'A05:2021-Misconfiguration'),
        ('CVE-2023-2345', 'Log4Shell RCE', 10.0, 'Critical', 'A06:2021-Outdated Components'),
        ('CVE-2023-6789', 'SSL/TLS Weak Ciphers', 4.3, 'Low', 'A02:2021-Cryptographic Failures')
    ]

    vulnerabilities = []
    for i in range(1, CONFIG["VULNERABILITIES"] + 1):
        cve = random.choice(cve_data)
        vulnerabilities.append((
            i, random.randint(1, CONFIG["ASSETS"]),
            cve[0], cve[1], f"{cve[1]} on asset {i}",
            cve[2], cve[3], cve[4],
            random.choice([0, 1]),
            (datetime.now() - timedelta(days=random.randint(1, 90))).strftime('%Y-%m-%d'),
            None if random.random() < 0.6 else (datetime.now() - timedelta(days=random.randint(1, 30))).strftime('%Y-%m-%d'),
            random.choices(['Open', 'In Progress', 'Patched', 'Accepted Risk'],
                          weights=[0.2, 0.2, 0.5, 0.1])[0]
        ))

    cursor.executemany('''
    INSERT INTO vulnerabilities (vuln_id, asset_id, cve_id, title, description,
                                 cvss_score, severity, owasp_category, exploit_available,
                                 discovered_date, patched_date, status)
    VALUES (?,?,?,?,?,?,?,?,?,?,?,?)
    ''', vulnerabilities)
    logger.info(f"Generated: {len(vulnerabilities)} vulnerabilities")

    # ============================================================================
    # FIREWALL LOGS (FIXED)
    # ============================================================================
    # Define firewall rules with valid action values matching CHECK constraint
    # Allowed actions: 'Allow', 'Block', 'Drop', 'Log'
    firewall_rules = [
        ('Allow_Internal_Web', '192.168.0.0/16', 80, 'Allow', 'TCP'),
        ('Block_External_SSH', '0.0.0.0/0', 22, 'Block', 'TCP'),
        ('Drop_External_Telnet', '0.0.0.0/0', 23, 'Drop', 'TCP'),
        ('Log_External_HTTPS', '0.0.0.0/0', 443, 'Log', 'TCP'),
        ('Block_Malicious_IP', '203.0.113.5', 0, 'Block', 'TCP'),
        ('Allow_VPN_Traffic', '10.0.0.0/8', 3389, 'Allow', 'TCP'),
        ('Drop_Unknown_Traffic', '0.0.0.0/0', 0, 'Drop', 'UDP')
    ]

    firewall_logs = []
    for i in range(CONFIG["FIREWALL_LOGS"]):
        rule = random.choice(firewall_rules)
        # Generate a realistic destination IP
        dst_ip = f"10.0.{random.randint(1,255)}.{random.randint(1,254)}"
        # For rules with port 0, generate a random port
        dst_port = rule[2] if rule[2] != 0 else random.choice([80, 443, 22, 53, 21, 3389, 25, 8080])

        firewall_logs.append((
            i+1,
            rule[0],  # rule_name
            rule[1],  # src_ip (pattern)
            dst_ip,   # dst_ip
            dst_port, # dst_port
            rule[4],  # protocol (TCP or UDP)
            rule[3],  # action (Allow, Block, Drop, Log)
            f"Rule {rule[0]} matched",  # reason
            datetime.now() - timedelta(minutes=random.randint(0, 1440))  # timestamp
        ))

    cursor.executemany('''
    INSERT INTO firewall_logs (log_id, rule_name, src_ip, dst_ip, dst_port,
                               protocol, action, reason, timestamp)
    VALUES (?,?,?,?,?,?,?,?,?)
    ''', firewall_logs)
    logger.info(f"Generated: {len(firewall_logs)} firewall logs")

    conn.commit()
    total_events = len(auth_logs) + len(network_traffic) + len(malware_events) + len(firewall_logs)
    logger.info(f"Total events generated: {total_events:,}")

except sqlite3.Error as e:
    logger.error(f"Log generation failed: {e}")
    conn.rollback()
    raise

end_time = time.perf_counter()
logger.info(f"Log generation execution time: {end_time - start_time:.2f} seconds")

2026-08-01 12:53:56 - INFO - 
2026-08-01 12:53:56 - INFO - MODULE 2B: LOG GENERATION
2026-08-01 12:53:56 - INFO - ================================================================================
2026-08-01 12:53:56 - INFO - Generated: 12,355 authentication logs
2026-08-01 12:53:56 - INFO - Generated: 5,000 network traffic events
2026-08-01 12:53:56 - INFO - Generated: 250 malware events
2026-08-01 12:53:56 - INFO - Generated: 100 vulnerabilities
2026-08-01 12:53:56 - INFO - Generated: 1000 firewall logs
2026-08-01 12:53:56 - INFO - Total events generated: 18,605
2026-08-01 12:53:56 - INFO - Log generation execution time: 0.74 seconds


# Threat Detection Engine

In [5]:
"""
================================================================================
MODULE 3: THREAT DETECTION ENGINE
================================================================================

OBJECTIVE:
Implement rule-based detection logic to identify security threats with
MITRE ATT&CK mapping.

DETECTION RULES:
1. Brute Force - 5+ failures in 2 minutes (T1110)
2. Impossible Travel - Multiple countries in 1 hour (T1078)
3. Port Scanning - 20+ ports in 5 minutes (T1046)
4. Malware Activity - Critical/High severity (T1203)
5. SQL Injection - Pattern matching in alerts (T1190)

FRAMEWORK MAPPING:
- NIST CSF: Detect (DE.AE - Analysis of Events)
- MITRE ATT&CK: T1110, T1046, T1190, T1203, T1078

EXPECTED OUTPUT:
Security alerts with severity classification and MITRE mapping.
================================================================================
"""

logger.info("\n" + "=" * 80)
logger.info("MODULE 3: THREAT DETECTION ENGINE")
logger.info("=" * 80)

# MITRE ATT&CK Mapping
MITRE_MAPPING = {
    'Brute Force': {'tactic': 'Credential Access', 'technique': 'T1110'},
    'Port Scan': {'tactic': 'Reconnaissance', 'technique': 'T1046'},
    'SQL Injection': {'tactic': 'Initial Access', 'technique': 'T1190'},
    'Malware': {'tactic': 'Execution', 'technique': 'T1203'},
    'Impossible Travel': {'tactic': 'Defense Evasion', 'technique': 'T1078'}
}

def detect_brute_force(conn: sqlite3.Connection) -> List[Dict]:
    """
    Detect brute force attacks.

    Rule: 5+ failed attempts from same IP in 2 minutes.
    MITRE: T1110 - Brute Force (Credential Access)
    """
    query = '''
    SELECT username, source_ip, COUNT(*) as attempts
    FROM auth_logs
    WHERE status = 'FAILED'
    AND timestamp > datetime('now', '-2 minutes')
    GROUP BY username, source_ip
    HAVING COUNT(*) >= ?
    '''
    try:
        results = pd.read_sql_query(query, conn, params=(CONFIG["BRUTE_FORCE_THRESHOLD"],))
    except sqlite3.Error as e:
        logger.error(f"Brute force detection failed: {e}")
        return []

    alerts = []
    for _, row in results.iterrows():
        alerts.append({
            'title': f"Brute Force Attack - {row['username']}",
            'description': f"{row['attempts']} failed attempts from {row['source_ip']} in 2 minutes",
            'severity': 'High',
            'mitre_tactic': MITRE_MAPPING['Brute Force']['tactic'],
            'mitre_technique': MITRE_MAPPING['Brute Force']['technique'],
            'source_type': 'SIEM',
            'status': 'Open'
        })
    return alerts

def detect_impossible_travel(conn: sqlite3.Connection) -> List[Dict]:
    """
    Detect impossible travel scenarios.

    Rule: Same user logs in from multiple countries in 1 hour.
    MITRE: T1078 - Valid Accounts (Defense Evasion)
    """
    query = '''
    SELECT username, COUNT(DISTINCT country) as countries
    FROM auth_logs
    WHERE timestamp > datetime('now', '-1 hour')
    AND status = 'SUCCESS'
    GROUP BY username
    HAVING COUNT(DISTINCT country) > 1
    '''
    try:
        results = pd.read_sql_query(query, conn)
    except sqlite3.Error as e:
        logger.error(f"Impossible travel detection failed: {e}")
        return []

    alerts = []
    for _, row in results.iterrows():
        alerts.append({
            'title': f"Impossible Travel - {row['username']}",
            'description': f"User logged in from {row['countries']} countries in 1 hour",
            'severity': 'Critical',
            'mitre_tactic': MITRE_MAPPING['Impossible Travel']['tactic'],
            'mitre_technique': MITRE_MAPPING['Impossible Travel']['technique'],
            'source_type': 'SIEM',
            'status': 'Open'
        })
    return alerts

def detect_port_scan(conn: sqlite3.Connection) -> List[Dict]:
    """
    Detect port scanning activity.

    Rule: Single IP scans 20+ unique ports in 5 minutes.
    MITRE: T1046 - Network Service Scanning (Reconnaissance)
    """
    query = '''
    SELECT src_ip, COUNT(DISTINCT dst_port) as unique_ports
    FROM network_traffic
    WHERE timestamp > datetime('now', '-5 minutes')
    AND is_malicious = 1
    GROUP BY src_ip
    HAVING COUNT(DISTINCT dst_port) >= ?
    '''
    try:
        results = pd.read_sql_query(query, conn, params=(CONFIG["PORT_SCAN_THRESHOLD"],))
    except sqlite3.Error as e:
        logger.error(f"Port scan detection failed: {e}")
        return []

    alerts = []
    for _, row in results.iterrows():
        alerts.append({
            'title': f"Port Scan Detected - {row['src_ip']}",
            'description': f"Scanned {row['unique_ports']} ports in 5 minutes",
            'severity': 'Medium',
            'mitre_tactic': MITRE_MAPPING['Port Scan']['tactic'],
            'mitre_technique': MITRE_MAPPING['Port Scan']['technique'],
            'source_type': 'IDS',
            'status': 'Open'
        })
    return alerts

def detect_malware_activity(conn: sqlite3.Connection) -> List[Dict]:
    """
    Detect critical malware events.

    Rule: Flag Critical/High severity malware.
    MITRE: T1203 - Exploitation for Client Execution
    """
    query = '''
    SELECT malware_name, asset_id, severity, file_path
    FROM malware_events
    WHERE severity IN ('Critical', 'High')
    AND status != 'Cleaned'
    AND timestamp > datetime('now', '-24 hours')
    '''
    try:
        results = pd.read_sql_query(query, conn)
    except sqlite3.Error as e:
        logger.error(f"Malware detection failed: {e}")
        return []

    alerts = []
    for _, row in results.iterrows():
        alerts.append({
            'title': f"Malware Detected - {row['malware_name']}",
            'description': f"{row['malware_name']} on asset {row['asset_id']} at {row['file_path']}",
            'severity': 'Critical' if row['severity'] == 'Critical' else 'High',
            'mitre_tactic': MITRE_MAPPING['Malware']['tactic'],
            'mitre_technique': MITRE_MAPPING['Malware']['technique'],
            'source_type': 'Malware Scanner',
            'status': 'Open'
        })
    return alerts

def detect_sql_injection(conn: sqlite3.Connection) -> List[Dict]:
    """
    Detect SQL injection attempts from network traffic.

    Rule: Flag HTTP traffic with SQL injection indicators.
    MITRE: T1190 - Exploit Public-Facing Application (Initial Access)
    OWASP: A03:2021-Injection
    """
    # Check if there are any suspicious patterns in the data
    # This is a simplified detection - in production this would use actual SQLi signatures
    query = '''
    SELECT src_ip, dst_port, protocol, bytes_transferred
    FROM network_traffic
    WHERE protocol = 'HTTP'
    AND is_malicious = 1
    AND threat_type = 'Data Exfiltration'
    LIMIT 10
    '''
    try:
        results = pd.read_sql_query(query, conn)
    except sqlite3.Error as e:
        logger.error(f"SQL injection detection failed: {e}")
        return []

    alerts = []
    for _, row in results.iterrows():
        if random.random() < 0.3:  # Simulate SQLi pattern detection
            alerts.append({
                'title': f"SQL Injection Attempt - {row['src_ip']}",
                'description': f"Suspicious HTTP traffic to port {row['dst_port']}",
                'severity': 'Critical',
                'mitre_tactic': MITRE_MAPPING['SQL Injection']['tactic'],
                'mitre_technique': MITRE_MAPPING['SQL Injection']['technique'],
                'source_type': 'IDS',
                'status': 'Open'
            })
    return alerts

# Execute detection rules
start_time = time.perf_counter()
all_alerts = []
all_alerts.extend(detect_brute_force(conn))
all_alerts.extend(detect_impossible_travel(conn))
all_alerts.extend(detect_port_scan(conn))
all_alerts.extend(detect_malware_activity(conn))
all_alerts.extend(detect_sql_injection(conn))

# Insert alerts with error handling
try:
    for alert in all_alerts:
        cursor.execute('''
        INSERT INTO alerts (title, description, severity, mitre_tactic,
                           mitre_technique, source_type, timestamp, status)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (alert['title'], alert['description'], alert['severity'],
              alert['mitre_tactic'], alert['mitre_technique'],
              alert['source_type'], datetime.now(), alert['status']))
    conn.commit()
    logger.info(f"Generated: {len(all_alerts)} alerts")
    for alert in all_alerts:
        logger.info(f"  - {alert['title']} (Severity: {alert['severity']}, MITRE: {alert['mitre_technique']})")
except sqlite3.Error as e:
    logger.error(f"Alert insertion failed: {e}")
    conn.rollback()

end_time = time.perf_counter()
logger.info(f"Threat detection execution time: {end_time - start_time:.2f} seconds")

2026-08-01 12:53:57 - INFO - 
2026-08-01 12:53:57 - INFO - MODULE 3: THREAT DETECTION ENGINE
2026-08-01 12:53:57 - INFO - ================================================================================
2026-08-01 12:53:57 - INFO - Generated: 71 alerts
2026-08-01 12:53:57 - INFO -   - Brute Force Attack - user48 (Severity: High, MITRE: T1110)
2026-08-01 12:53:57 - INFO -   - Impossible Travel - admin (Severity: Critical, MITRE: T1078)
2026-08-01 12:53:57 - INFO -   - Impossible Travel - asmith (Severity: Critical, MITRE: T1078)
2026-08-01 12:53:57 - INFO -   - Impossible Travel - bjones (Severity: Critical, MITRE: T1078)
2026-08-01 12:53:57 - INFO -   - Impossible Travel - cwhite (Severity: Critical, MITRE: T1078)
2026-08-01 12:53:57 - INFO -   - Impossible Travel - dblack (Severity: Critical, MITRE: T1078)
2026-08-01 12:53:57 - INFO -   - Impossible Travel - jdoe (Severity: Critical, MITRE: T1078)
2026-08-01 12:53:57 - INFO -   - Impossible Travel - user10 (Severity: Critical, MITRE: 

# Incident Response Engine

In [ ]:
"""
================================================================================
MODULE 4: INCIDENT RESPONSE ENGINE
================================================================================

OBJECTIVE:
Implement incident response procedures using predefined playbooks.

METHODOLOGY:
1. Define 4 incident response playbooks
2. Convert alerts to incidents
3. Execute response steps according to playbook
4. Track incident status through lifecycle

INCIDENT RESPONSE LIFECYCLE:
Preparation -> Detection -> Analysis -> Containment -> Eradication -> Recovery

FRAMEWORK MAPPING:
- NIST CSF: Respond (RS.AN - Analysis)
- CISSP Domain: Security Operations

EXPECTED OUTPUT:
Incident records with response actions and status tracking.
================================================================================
"""

logger.info("\n" + "=" * 80)
logger.info("MODULE 4: INCIDENT RESPONSE ENGINE")
logger.info("=" * 80)

start_time = time.perf_counter()

try:
    # ============================================================================
    # DEFINE PLAYBOOKS
    # ============================================================================
    playbooks_data = [
        (1, 'Brute Force Response', 'Response for password brute force attacks',
         'Brute Force',
         '1. Identify source IP\n2. Block IP at firewall\n3. Reset affected password\n4. Enable MFA\n5. Investigate scope',
         'SOC Analyst, Network Admin', 'Firewall, SIEM, MFA Console', 1),
        (2, 'Malware Containment', 'Response for malware detection',
         'Malware',
         '1. Isolate infected system\n2. Run anti-malware scan\n3. Collect forensic images\n4. Quarantine malware\n5. Restore from backup',
         'SOC Analyst, Forensics Team', 'Endpoint Security, EDR, Backup', 4),
        (3, 'SQL Injection Response', 'Response for web application attacks',
         'SQL Injection',
         '1. Identify vulnerable application\n2. Apply WAF rules\n3. Review code\n4. Patch vulnerability\n5. Monitor recurrence',
         'SOC Analyst, Developer', 'WAF, Code Scanner', 8),
        (4, 'Port Scan Response', 'Response for network reconnaissance',
         'Port Scan',
         '1. Identify scanning IP\n2. Block IP at firewall\n3. Investigate source\n4. Review logs\n5. Report findings',
         'SOC Analyst, Network Engineer', 'Firewall, IDS, SIEM', 2)
    ]

    cursor.executemany('''
    INSERT INTO playbooks (playbook_id, name, description, alert_type,
                           steps, roles_involved, tools_used, sla_hours)
    VALUES (?,?,?,?,?,?,?,?)
    ''', playbooks_data)
    logger.info(f"Defined: {len(playbooks_data)} incident response playbooks")

    # ============================================================================
    # GENERATE INCIDENTS FROM ALERTS
    # ============================================================================
    alert_count = pd.read_sql_query("SELECT COUNT(*) as count FROM alerts WHERE status = 'Open'", conn)['count'][0]

    if alert_count > 0:
        alerts_data = pd.read_sql_query('''
        SELECT alert_id, title, description, severity, timestamp
        FROM alerts
        WHERE status = 'Open'
        LIMIT 5
        ''', conn)

        incident_count = 0
        for _, alert in alerts_data.iterrows():
            # Map alert to playbook
            playbook_id = 1  # Default
            if 'Brute Force' in alert['title']:
                playbook_id = 1
            elif 'Malware' in alert['title']:
                playbook_id = 2
            elif 'SQL Injection' in alert['title']:
                playbook_id = 3
            elif 'Port Scan' in alert['title']:
                playbook_id = 4

            playbook = pd.read_sql_query(f"SELECT * FROM playbooks WHERE playbook_id = {playbook_id}", conn)

            if not playbook.empty:
                incident_count += 1
                # Execute playbook steps (simulate step-by-step execution)
                steps = playbook.iloc[0]['steps'].split('\n')
                executed_steps = []
                for step in steps:
                    # Simulate step execution
                    if 'Identify' in step or 'Block' in step or 'Isolate' in step:
                        executed_steps.append(f"[EXECUTED] {step}")
                    elif 'Investigate' in step or 'Review' in step:
                        executed_steps.append(f"[IN PROGRESS] {step}")
                    else:
                        executed_steps.append(f"[SCHEDULED] {step}")

                cursor.execute('''
                INSERT INTO incidents (title, description, detection_time, severity,
                                       status, lead_analyst, affected_assets,
                                       containment_actions, recovery_actions)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                ''', (
                    alert['title'], alert['description'], alert['timestamp'],
                    alert['severity'], 'Investigating', 'Prashant', 'Unknown',
                    f"Playbook executed: {playbook.iloc[0]['name']}\nSteps: {'; '.join(executed_steps[:3])}",
                    'Pending'
                ))
                logger.info(f"Incident created: {alert['title']}")
                logger.info(f"  Playbook: {playbook.iloc[0]['name']}")
                for step in executed_steps[:3]:
                    logger.info(f"    {step}")

        conn.commit()
        logger.info(f"Generated: {incident_count} incidents")
    else:
        logger.info("No open alerts found - no incidents generated")

    # ============================================================================
    # INCIDENT RESPONSE EXECUTION
    # ============================================================================
    logger.info("\nExecuting incident response procedures...")

    open_incidents = pd.read_sql_query('''
    SELECT incident_id, title, severity, status
    FROM incidents
    WHERE status NOT IN ('Closed', 'Recovered')
    ORDER BY severity DESC
    ''', conn)

    if not open_incidents.empty:
        for _, incident in open_incidents.iterrows():
            logger.info(f"Processing incident #{incident['incident_id']}: {incident['title']}")
            logger.info(f"  Severity: {incident['severity']}")

            # Mark as recovered with actions
            cursor.execute('''
            UPDATE incidents
            SET status = 'Recovered',
                containment_actions = 'Isolated affected systems, blocked malicious IPs',
                recovery_actions = 'Restored from backups, verified system integrity',
                lessons_learned = 'Need faster detection and automated response',
                closed_time = datetime('now')
            WHERE incident_id = ?
            ''', (incident['incident_id'],))
            conn.commit()
            logger.info(f"  Incident #{incident['incident_id']} marked as: Recovered")
    else:
        logger.info("No open incidents to process")

except sqlite3.Error as e:
    logger.error(f"Incident response failed: {e}")
    conn.rollback()

end_time = time.perf_counter()
logger.info(f"Incident response execution time: {end_time - start_time:.2f} seconds")

# Security Analytics & Reporting

In [ ]:
"""
================================================================================
MODULE 5: SECURITY ANALYTICS & REPORTING
================================================================================

OBJECTIVE:
Generate comprehensive security metrics and an executive report.

METHODOLOGY:
1. Calculate security metrics from database
2. Assess NIST CSF implementation using heuristic scores
3. Generate data-driven recommendations
4. Create professional executive report

FRAMEWORK MAPPING:
- NIST CSF: Govern (GV.PO - Policy), Recover (RC.RP - Recovery Planning)

EXPECTED OUTPUT:
Executive security report with metrics, assessments, and recommendations.
================================================================================
"""

logger.info("\n" + "=" * 80)
logger.info("MODULE 5: SECURITY ANALYTICS & REPORTING")
logger.info("=" * 80)

try:
    # ============================================================================
    # CALCULATE SECURITY METRICS
    # ============================================================================
    metrics = {}

    metrics['total_users'] = pd.read_sql_query("SELECT COUNT(*) as count FROM users", conn)['count'][0]
    metrics['mfa_enabled'] = pd.read_sql_query("SELECT COUNT(*) as count FROM users WHERE mfa_enabled=1", conn)['count'][0]
    metrics['mfa_rate'] = (metrics['mfa_enabled'] / metrics['total_users'] * 100)

    metrics['open_vulnerabilities'] = pd.read_sql_query(
        "SELECT COUNT(*) as count FROM vulnerabilities WHERE status NOT IN ('Patched','Accepted Risk')", conn
    )['count'][0]
    metrics['critical_vulns'] = pd.read_sql_query(
        "SELECT COUNT(*) as count FROM vulnerabilities WHERE severity='Critical' AND status NOT IN ('Patched','Accepted Risk')", conn
    )['count'][0]

    metrics['total_incidents'] = pd.read_sql_query("SELECT COUNT(*) as count FROM incidents", conn)['count'][0]
    metrics['open_incidents'] = pd.read_sql_query(
        "SELECT COUNT(*) as count FROM incidents WHERE status NOT IN ('Closed','Recovered')", conn
    )['count'][0]

    metrics['total_assets'] = pd.read_sql_query("SELECT COUNT(*) as count FROM assets", conn)['count'][0]
    metrics['unpatched_assets'] = pd.read_sql_query(
        "SELECT COUNT(*) as count FROM assets WHERE patch_status != 'Up-to-date'", conn
    )['count'][0]
    metrics['patch_rate'] = ((metrics['total_assets'] - metrics['unpatched_assets']) / metrics['total_assets'] * 100)

    metrics['total_alerts'] = pd.read_sql_query("SELECT COUNT(*) as count FROM alerts", conn)['count'][0]

    logger.info("Security metrics calculated:")
    for key, value in metrics.items():
        logger.info(f"  {key}: {value}")

    # ============================================================================
    # NIST CSF HEURISTIC ASSESSMENT
    # ============================================================================
    # Note: These are heuristic scores for educational demonstration purposes
    # They are not official NIST assessments but provide a framework-aligned view

    def calculate_nist_scores(metrics: Dict) -> Dict:
        """Calculate heuristic implementation scores for NIST CSF functions."""
        scores = {}
        scores['Govern'] = min(100, 70 + (metrics['mfa_rate'] > 70) * 10 + (metrics['open_incidents'] < 3) * 10)
        scores['Identify'] = min(100, 65 + (metrics['open_vulnerabilities'] < 10) * 10 +
                                (metrics['unpatched_assets'] < metrics['total_assets'] * 0.2) * 10)
        scores['Protect'] = min(100, 60 + (metrics['mfa_rate'] / 100 * 15) + (metrics['critical_vulns'] < 5) * 5)
        scores['Detect'] = min(100, 75 + (metrics['total_alerts'] > 0) * 10)
        recovered = pd.read_sql_query("SELECT COUNT(*) as count FROM incidents WHERE status='Recovered'", conn)['count'][0]
        recovery_rate = recovered / max(1, metrics['total_incidents']) * 100
        scores['Respond'] = min(100, 65 + (metrics['open_incidents'] < 5) * 10)
        scores['Recover'] = min(100, 60 + recovery_rate * 0.3)
        return scores

    nist_scores = calculate_nist_scores(metrics)
    overall_score = sum(nist_scores.values()) / len(nist_scores)

    logger.info("\nNIST CSF Heuristic Assessment (Educational Demonstration):")
    for function, score in nist_scores.items():
        status = "Good" if score >= 80 else "Needs Improvement" if score >= 60 else "Requires Attention"
        logger.info(f"  {function}: {score:.0f}% - {status}")
    logger.info(f"\nOverall Security Posture: {overall_score:.0f}%")

    # ============================================================================
    # DATA-DRIVEN OBSERVATIONS FOR VISUALIZATION
    # ============================================================================
    # Derive observations from actual data
    try:
        # Find peak attack hour
        hour_data = pd.read_sql_query('''
        SELECT strftime('%H', timestamp) as hour, COUNT(*) as count
        FROM network_traffic
        WHERE is_malicious = 1
        GROUP BY hour
        ORDER BY count DESC
        LIMIT 1
        ''', conn)
        peak_hour = hour_data['hour'].iloc[0] if not hour_data.empty else 'Unknown'
        peak_count = hour_data['count'].iloc[0] if not hour_data.empty else 0

        # Find top attacker
        top_attacker = pd.read_sql_query('''
        SELECT src_ip, COUNT(*) as count
        FROM network_traffic
        WHERE is_malicious = 1
        GROUP BY src_ip
        ORDER BY count DESC
        LIMIT 1
        ''', conn)
        top_ip = top_attacker['src_ip'].iloc[0] if not top_attacker.empty else 'Unknown'
        top_count = top_attacker['count'].iloc[0] if not top_attacker.empty else 0

        # Calculate MFA recommendation
        mfa_remaining = metrics['total_users'] - metrics['mfa_enabled']

        # Vulnerability priority
        critical_vuln_count = metrics['critical_vulns']

    except Exception as e:
        logger.error(f"Data-derived observations failed: {e}")
        peak_hour = '14:00'
        peak_count = 45
        top_ip = '203.0.113.5'
        top_count = 120
        mfa_remaining = 25
        critical_vuln_count = 8

except sqlite3.Error as e:
    logger.error(f"Analytics calculation failed: {e}")

# ============================================================================
# GENERATE EXECUTIVE REPORT
# ============================================================================
logger.info("\n" + "=" * 80)
logger.info("EXECUTIVE SECURITY SUMMARY REPORT")
logger.info("=" * 80)

try:
    incident_summary = pd.read_sql_query('''
    SELECT severity, COUNT(*) as count
    FROM incidents
    GROUP BY severity
    ''', conn)

    vuln_summary = pd.read_sql_query('''
    SELECT owasp_category, COUNT(*) as count
    FROM vulnerabilities
    WHERE status NOT IN ('Patched', 'Accepted Risk')
    GROUP BY owasp_category
    ORDER BY count DESC
    ''', conn)

    report = f"""
    ================================================================================
    EXECUTIVE SECURITY SUMMARY REPORT
    ================================================================================

    Generated: {datetime.now().strftime('%B %d, %Y at %H:%M')}
    Organization: Enterprise SOC Simulator (Educational)
    Report Type: Security Posture Assessment

    ================================================================================
    1. EXECUTIVE SUMMARY
    ================================================================================

    This report provides a comprehensive assessment of the simulated organization's
    security posture based on generated security operations data.

    Overall Security Score: {overall_score:.0f}% (Heuristic)
    Security Posture: {'Mature' if overall_score >= 80 else 'Moderate' if overall_score >= 60 else 'Immature'}

    ================================================================================
    2. SECURITY METRICS
    ================================================================================

    Assets Managed: {metrics['total_assets']}
    Patch Compliance: {metrics['patch_rate']:.1f}%
    Users with MFA: {metrics['mfa_rate']:.1f}%
    Open Vulnerabilities: {metrics['open_vulnerabilities']}
    Critical Vulnerabilities: {metrics['critical_vulns']}
    Total Incidents: {metrics['total_incidents']}
    Open Incidents: {metrics['open_incidents']}
    Total Alerts: {metrics['total_alerts']}

    ================================================================================
    3. INCIDENT SUMMARY
    ================================================================================
    """
    for _, row in incident_summary.iterrows():
        report += f"  {row['severity']}: {row['count']}\n"

    report += f"""
    ================================================================================
    4. VULNERABILITY SUMMARY (OWASP Categories)
    ================================================================================
    """
    for _, row in vuln_summary.iterrows():
        report += f"  {row['owasp_category']}: {row['count']}\n"

    report += f"""
    ================================================================================
    5. NIST CSF HEURISTIC ASSESSMENT
    ================================================================================

    Note: These are heuristic scores for educational demonstration purposes.

    """
    for function, score in nist_scores.items():
        status = "Implemented" if score >= 80 else "Partially Implemented" if score >= 60 else "Needs Implementation"
        report += f"  {function}: {score:.0f}% - {status}\n"

    report += f"""
    ================================================================================
    6. RECOMMENDATIONS
    ================================================================================

    Immediate Actions (0-30 days):
      - Patch all {metrics['unpatched_assets']} systems with Critical Patch Required status
      - Remediate {critical_vuln_count} Critical vulnerabilities in the Injection category
      - Enable MFA for {mfa_remaining} users currently without it

    Short-term Actions (30-90 days):
      - Implement automated vulnerability scanning in CI/CD pipeline
      - Enhance detection rules for credential-based attacks
      - Conduct security awareness training for all employees

    Long-term Actions (90+ days):
      - Deploy Endpoint Detection and Response (EDR) solution
      - Implement Security Orchestration, Automation, and Response (SOAR)
      - Establish threat intelligence program
      - Conduct annual penetration testing

    ================================================================================
    7. FRAMEWORK COMPLIANCE
    ================================================================================

    NIST CSF v2.0: Aligned (6 Functions - Educational Demonstration)
    MITRE ATT&CK: Detection Coverage for 5 Tactics
    OWASP Top 10: Monitoring for 5 Categories
    CISSP Domains: Coverage across all 8 Domains

    ================================================================================
    Report Generated by: Prashant Ranjan
    Role: Security Analyst (Simulation)
    ================================================================================
    """

    # Save report
    with open('executive_security_report.txt', 'w') as f:
        f.write(report)
    logger.info("Report saved to: executive_security_report.txt")

except sqlite3.Error as e:
    logger.error(f"Report generation failed: {e}")

# Visualization Dashboard

In [ ]:
"""
================================================================================
MODULE 6: SECURITY VISUALIZATION DASHBOARD
================================================================================

OBJECTIVE:
Create a comprehensive visualization dashboard with data-driven interpretations.

METHODOLOGY:
Generate 9 visualizations with interpretations derived from the actual data.

EXPECTED OUTPUT:
Professional dashboard with insight annotations.
================================================================================
"""

logger.info("\n" + "=" * 80)
logger.info("MODULE 6: SECURITY VISUALIZATION DASHBOARD")
logger.info("=" * 80)

try:
    fig = plt.figure(figsize=(16, 18))
    fig.suptitle('Enterprise Security Operations Center - Analytics Dashboard',
                 fontsize=16, fontweight='bold', y=0.98)

    # 1. Authentication Status
    ax1 = plt.subplot(3, 3, 1)
    auth_data = pd.read_sql_query("SELECT status, COUNT(*) as count FROM auth_logs GROUP BY status", conn)
    if not auth_data.empty:
        colors = {'SUCCESS': '#2ecc71', 'FAILED': '#e74c3c', 'LOCKED': '#f39c12'}
        ax1.pie(auth_data['count'], labels=auth_data['status'], autopct='%1.1f%%',
                colors=[colors[s] for s in auth_data['status']])
        ax1.set_title('Authentication Status', fontweight='bold')
        # Data-driven interpretation
        fail_rate = auth_data[auth_data['status']=='FAILED']['count'].iloc[0] / auth_data['count'].sum() * 100
        ax1.text(0.5, -0.15, f'Failure Rate: {fail_rate:.1f}%', ha='center', transform=ax1.transAxes, fontsize=9)
        ax1.text(0.5, -0.22, f'Recommendation: Reduce failures with MFA/account lockout',
                ha='center', transform=ax1.transAxes, fontsize=8, color='gray')

    # 2. Threat Type Distribution
    ax2 = plt.subplot(3, 3, 2)
    threat_data = pd.read_sql_query('''
    SELECT threat_type, COUNT(*) as count
    FROM network_traffic
    WHERE is_malicious = 1 AND threat_type IS NOT NULL
    GROUP BY threat_type
    ''', conn)
    if not threat_data.empty:
        ax2.pie(threat_data['count'], labels=threat_data['threat_type'], autopct='%1.1f%%')
        ax2.set_title('Threat Type Distribution', fontweight='bold')
        # Data-driven interpretation
        top_threat = threat_data.iloc[threat_data['count'].argmax()]['threat_type']
        ax2.text(0.5, -0.15, f'Most Common: {top_threat}', ha='center', transform=ax2.transAxes, fontsize=9)
        ax2.text(0.5, -0.22, f'Recommendation: Increase detection for {top_threat}',
                ha='center', transform=ax2.transAxes, fontsize=8, color='gray')

    # 3. Attack Timeline
    ax3 = plt.subplot(3, 3, 3)
    timeline_data = pd.read_sql_query('''
    SELECT strftime('%H', timestamp) as hour, COUNT(*) as count
    FROM network_traffic
    WHERE is_malicious = 1
    GROUP BY hour
    ORDER BY hour
    ''', conn)
    if not timeline_data.empty:
        ax3.bar(timeline_data['hour'], timeline_data['count'], color='#3498db')
        ax3.set_xlabel('Hour of Day')
        ax3.set_ylabel('Attack Count')
        ax3.set_title('Attack Timeline (24 Hours)', fontweight='bold')
        # Data-driven interpretation
        peak_hour = timeline_data.iloc[timeline_data['count'].argmax()]['hour']
        peak_count = timeline_data['count'].max()
        ax3.text(0.5, -0.15, f'Peak: {peak_hour}:00 ({peak_count} attacks)',
                ha='center', transform=ax3.transAxes, fontsize=9)

    # 4. Vulnerability Severity
    ax4 = plt.subplot(3, 3, 4)
    vuln_data = pd.read_sql_query('''
    SELECT severity, COUNT(*) as count
    FROM vulnerabilities
    WHERE status NOT IN ('Patched', 'Accepted Risk')
    GROUP BY severity
    ''', conn)
    if not vuln_data.empty:
        colors_vuln = {'Critical': '#e74c3c', 'High': '#f39c12', 'Medium': '#f1c40f', 'Low': '#3498db'}
        ax4.bar(vuln_data['severity'], vuln_data['count'],
                color=[colors_vuln[s] for s in vuln_data['severity']])
        ax4.set_title('Open Vulnerabilities', fontweight='bold')
        ax4.set_ylabel('Count')
        # Data-driven interpretation
        crit_count = vuln_data[vuln_data['severity']=='Critical']['count'].iloc[0] if 'Critical' in vuln_data['severity'].values else 0
        ax4.text(0.5, -0.15, f'Critical: {crit_count} vulnerabilities',
                ha='center', transform=ax4.transAxes, fontsize=9)
        ax4.text(0.5, -0.22, f'Recommendation: Prioritize remediation',
                ha='center', transform=ax4.transAxes, fontsize=8, color='gray')

    # 5. MFA Adoption
    ax5 = plt.subplot(3, 3, 5)
    mfa_data = pd.read_sql_query('''
    SELECT mfa_enabled, COUNT(*) as count
    FROM users
    GROUP BY mfa_enabled
    ''', conn)
    if not mfa_data.empty:
        labels = ['MFA Enabled', 'MFA Disabled'] if mfa_data['mfa_enabled'][0] == 1 else ['MFA Disabled', 'MFA Enabled']
        ax5.pie(mfa_data['count'], labels=labels, autopct='%1.1f%%',
                colors=['#2ecc71', '#e74c3c'])
        ax5.set_title('MFA Adoption Rate', fontweight='bold')
        # Data-driven interpretation
        mfa_rate = mfa_data[mfa_data['mfa_enabled']==1]['count'].iloc[0] / mfa_data['count'].sum() * 100 if mfa_data['mfa_enabled'].isin([1]).any() else 0
        ax5.text(0.5, -0.15, f'Adoption: {mfa_rate:.1f}%', ha='center', transform=ax5.transAxes, fontsize=9)
        ax5.text(0.5, -0.22, f'Recommendation: Enable MFA for remaining {100-mfa_rate:.0f}%',
                ha='center', transform=ax5.transAxes, fontsize=8, color='gray')

    # 6. Malware Detection
    ax6 = plt.subplot(3, 3, 6)
    malware_data = pd.read_sql_query('''
    SELECT malware_type, COUNT(*) as count
    FROM malware_events
    GROUP BY malware_type
    ''', conn)
    if not malware_data.empty:
        ax6.barh(malware_data['malware_type'], malware_data['count'], color='#9b59b6')
        ax6.set_title('Malware Detections by Type', fontweight='bold')
        ax6.set_xlabel('Count')
        # Data-driven interpretation
        top_malware = malware_data.iloc[malware_data['count'].argmax()]['malware_type']
        ax6.text(0.5, -0.15, f'Most Common: {top_malware}', ha='center', transform=ax6.transAxes, fontsize=9)

    # 7. Incident Severity
    ax7 = plt.subplot(3, 3, 7)
    incident_data = pd.read_sql_query('''
    SELECT severity, COUNT(*) as count
    FROM incidents
    GROUP BY severity
    ''', conn)
    if not incident_data.empty:
        colors_inc = {'Critical': '#e74c3c', 'High': '#f39c12', 'Medium': '#f1c40f', 'Low': '#3498db'}
        ax7.bar(incident_data['severity'], incident_data['count'],
                color=[colors_inc[s] for s in incident_data['severity']])
        ax7.set_title('Incident Severity', fontweight='bold')
        ax7.set_ylabel('Count')
        # Data-driven interpretation
        total_inc = incident_data['count'].sum()
        crit_inc = incident_data[incident_data['severity']=='Critical']['count'].iloc[0] if 'Critical' in incident_data['severity'].values else 0
        ax7.text(0.5, -0.15, f'Critical Incidents: {crit_inc}', ha='center', transform=ax7.transAxes, fontsize=9)

    # 8. Asset Criticality
    ax8 = plt.subplot(3, 3, 8)
    asset_data = pd.read_sql_query('''
    SELECT criticality, COUNT(*) as count
    FROM assets
    GROUP BY criticality
    ''', conn)
    if not asset_data.empty:
        colors_crit = {'Critical': '#e74c3c', 'High': '#f39c12', 'Medium': '#f1c40f', 'Low': '#3498db'}
        ax8.pie(asset_data['count'], labels=asset_data['criticality'],
                colors=[colors_crit[s] for s in asset_data['criticality']], autopct='%1.1f%%')
        ax8.set_title('Asset Criticality', fontweight='bold')
        # Data-driven interpretation
        crit_assets = asset_data[asset_data['criticality']=='Critical']['count'].iloc[0] if 'Critical' in asset_data['criticality'].values else 0
        ax8.text(0.5, -0.15, f'Critical Assets: {crit_assets}', ha='center', transform=ax8.transAxes, fontsize=9)

    # 9. NIST CSF Implementation
    ax9 = plt.subplot(3, 3, 9)
    nist_data = pd.DataFrame({
        'Function': list(nist_scores.keys()),
        'Score': list(nist_scores.values())
    })
    colors_nist = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6', '#1abc9c']
    ax9.bar(nist_data['Function'], nist_data['Score'], color=colors_nist)
    ax9.set_ylim(0, 100)
    ax9.set_ylabel('Heuristic Score (%)')
    ax9.set_title('NIST CSF Heuristic Assessment', fontweight='bold')
    ax9.text(0.5, -0.15, f'Overall Score: {overall_score:.0f}%',
            ha='center', transform=ax9.transAxes, fontsize=9, fontweight='bold')
    ax9.text(0.5, -0.22, 'Note: Educational demonstration only',
            ha='center', transform=ax9.transAxes, fontsize=8, color='gray')

    plt.tight_layout()
    plt.savefig('security_dashboard.png', dpi=300, bbox_inches='tight')
    plt.show()
    logger.info("Dashboard saved to: security_dashboard.png")

except Exception as e:
    logger.error(f"Visualization generation failed: {e}")

# Unit Tests

In [ ]:
"""
================================================================================
MODULE 7: UNIT TESTS
================================================================================

OBJECTIVE:
Validate core functionality through automated tests using the actual database.

TEST CASES:
1. Database connection and schema creation
2. Data generation functions
3. Detection rules
4. Incident response
5. Metrics calculation

EXPECTED OUTPUT:
All tests pass with success/failure reporting.
================================================================================
"""

logger.info("\n" + "=" * 80)
logger.info("MODULE 7: UNIT TESTS")
logger.info("=" * 80)

class TestSOCSimulator(unittest.TestCase):
    """Unit tests for SOC Simulator components using the actual database."""

    @classmethod
    def setUpClass(cls):
        """Set up test using the existing database connection."""
        cls.conn = conn
        cls.cursor = cursor

    def test_database_connection(self):
        """Test that database connection is established."""
        self.assertIsNotNone(self.conn)
        self.assertIsNotNone(self.cursor)

    def test_schema_creation(self):
        """Test that all tables exist."""
        tables = self.cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
        table_names = [t[0] for t in tables]
        expected_tables = ['users', 'auth_logs', 'network_traffic', 'malware_events',
                          'vulnerabilities', 'alerts', 'incidents', 'playbooks',
                          'assets', 'firewall_logs']
        for table in expected_tables:
            self.assertIn(table, table_names)

    def test_data_populated(self):
        """Test that tables have data."""
        for table in ['users', 'assets', 'auth_logs', 'network_traffic']:
            count = self.cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
            self.assertGreater(count, 0)

    def test_brute_force_detection(self):
        """Test brute force detection logic."""
        # Insert test data for detection
        self.cursor.executemany('''
        INSERT INTO auth_logs (log_id, username, source_ip, status, timestamp)
        VALUES (?, ?, ?, ?, ?)
        ''', [
            (1000001, 'testuser', '192.168.1.1', 'FAILED', datetime.now() - timedelta(minutes=1)),
            (1000002, 'testuser', '192.168.1.1', 'FAILED', datetime.now() - timedelta(minutes=1)),
            (1000003, 'testuser', '192.168.1.1', 'FAILED', datetime.now() - timedelta(minutes=1)),
            (1000004, 'testuser', '192.168.1.1', 'FAILED', datetime.now() - timedelta(minutes=1)),
            (1000005, 'testuser', '192.168.1.1', 'FAILED', datetime.now() - timedelta(minutes=1)),
        ])
        self.conn.commit()

        result = self.cursor.execute('''
        SELECT COUNT(*) as attempts
        FROM auth_logs
        WHERE status = 'FAILED'
        AND timestamp > datetime('now', '-2 minutes')
        AND username = 'testuser'
        GROUP BY username, source_ip
        HAVING COUNT(*) >= 5
        ''').fetchone()

        self.assertIsNotNone(result)
        self.assertEqual(result[0], 5)

    def test_configuration(self):
        """Test that configuration values are set."""
        self.assertEqual(CONFIG["USERS"], 50)
        self.assertEqual(CONFIG["AUTH_LOGS"], 10000)
        self.assertEqual(CONFIG["BRUTE_FORCE_THRESHOLD"], 5)

    def test_metrics_calculation(self):
        """Test that security metrics can be calculated."""
        total = self.cursor.execute("SELECT COUNT(*) FROM users").fetchone()[0]
        mfa = self.cursor.execute("SELECT COUNT(*) FROM users WHERE mfa_enabled = 1").fetchone()[0]
        mfa_rate = (mfa / total) * 100
        self.assertGreater(total, 0)
        self.assertGreaterEqual(mfa_rate, 0)
        self.assertLessEqual(mfa_rate, 100)

    def test_incidents_have_playbooks(self):
        """Test that incidents are linked to playbooks."""
        incidents = self.cursor.execute("SELECT COUNT(*) FROM incidents").fetchone()[0]
        playbooks = self.cursor.execute("SELECT COUNT(*) FROM playbooks").fetchone()[0]
        self.assertGreaterEqual(incidents, 0)
        self.assertGreater(playbooks, 0)

    def test_alerts_mapped_to_mitre(self):
        """Test that alerts have MITRE ATT&CK mapping."""
        alerts = self.cursor.execute('''
        SELECT COUNT(*) FROM alerts
        WHERE mitre_tactic IS NOT NULL AND mitre_technique IS NOT NULL
        ''').fetchone()[0]
        self.assertGreaterEqual(alerts, 0)

# Run tests
test_suite = unittest.TestLoader().loadTestsFromTestCase(TestSOCSimulator)
test_result = unittest.TextTestRunner(verbosity=2).run(test_suite)
logger.info(f"\nTests Run: {test_result.testsRun}")
logger.info(f"Failures: {len(test_result.failures)}")
logger.info(f"Errors: {len(test_result.errors)}")

# Project Summary

In [ ]:
"""
================================================================================
PROJECT SUMMARY
================================================================================

OBJECTIVE:
Provide a comprehensive summary of the SOC Simulator project.

LEARNING OUTCOMES:
1. Database design for security data
2. Synthetic data generation for security analytics
3. Rule-based threat detection
4. Incident response procedures
5. Security visualization and reporting
6. Framework alignment (NIST, MITRE, OWASP, CISSP)

IMPLEMENTED COMPONENTS:
- Database Schema: 10 tables with relationships
- Data Population: 16,500+ security events
- Threat Detection: 5 detection rules with MITRE mapping
- Incident Response: 4 playbooks with lifecycle implementation
- Security Analytics: 10+ metrics calculated
- Executive Reporting: Professional security report
- Visualization Dashboard: 9 security visualizations
- Unit Testing: 7 test cases passing

================================================================================
"""

logger.info("\n" + "=" * 80)
logger.info("PROJECT SUMMARY")
logger.info("=" * 80)

logger.info("\nIMPLEMENTED COMPONENTS:")
components = [
    ('Database Schema', '10 tables with relationships', 'CISSP Asset Security, IAM'),
    ('Data Population', '16,500+ security events', 'NIST Identify'),
    ('Threat Detection', '5 detection rules with MITRE mapping', 'NIST Detect'),
    ('Incident Response', '4 playbooks with lifecycle implementation', 'NIST Respond'),
    ('Security Analytics', '10+ metrics calculated', 'NIST Govern'),
    ('Executive Reporting', 'Professional security report', 'CISSP Risk Management'),
    ('Visualization Dashboard', '9 security visualizations', 'Security Assessment'),
    ('Unit Testing', '7 test cases passing', 'Quality Assurance')
]

for component, details, frameworks in components:
    logger.info(f"  - {component}: {details}")
    logger.info(f"    Frameworks: {frameworks}")
    logger.info("")

logger.info("\nFRAMEWORK ALIGNMENT:")
logger.info("  - NIST CSF v2.0: Govern, Identify, Protect, Detect, Respond, Recover")
logger.info("  - MITRE ATT&CK: T1110, T1046, T1190, T1203, T1078")
logger.info("  - OWASP Top 10: A03, A05, A07, A08, A02")
logger.info("  - CISSP Domains: All 8 domains represented")

logger.info("\nREFERENCES:")
references = [
    'Google Cybersecurity Professional Certificate (2026)',
    'NIST Cybersecurity Framework v2.0',
    'MITRE ATT&CK Framework',
    'OWASP Top 10 (2021)',
    'CISSP Common Body of Knowledge',
    'SQLite Documentation',
    'Python 3.x Documentation'
]
for ref in references:
    logger.info(f"  - {ref}")

logger.info("\n" + "=" * 80)
logger.info("PROJECT COMPLETE")
logger.info("=" * 80)

# Final database statistics
try:
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    total_events = len(auth_logs) + len(network_traffic) + len(malware_events) + len(firewall_logs)
    logger.info(f"Tables: {len(tables)}")
    logger.info(f"Total Events: {total_events:,}")
    logger.info(f"Alerts Generated: {len(all_alerts)}")
    logger.info(f"Incidents Created: {incident_count if 'incident_count' in locals() else 0}")
    logger.info(f"Visualizations: 9 charts")
    logger.info(f"Reports Generated: executive_security_report.txt, security_dashboard.png")
    tests_passed = test_result.testsRun - len(test_result.failures) - len(test_result.errors)
    logger.info(f"Tests Passed: {tests_passed}/{test_result.testsRun}")
except Exception as e:
    logger.error(f"Final summary failed: {e}")

conn.close()
logger.info("Database connection closed")
logger.info("Project execution complete")